<a href="https://colab.research.google.com/github/ahmadhalawanii/flyrank-ml-capstone/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmadhalawanii/flyrank-ml-capstone/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

**Lane: Freestyle — "Confound or Cause?"**

Week 4 froze a rule: rank pages by how far their CTR falls below the median CTR of their own
position tier. It scored **precision@50 = 0.860** against a **0.542** base rate — and its own
top-20 review found the head of that queue was one undifferentiated block of pages with zero
recorded clicks, where a title rewrite is the wrong action no matter how well the metric scores.

This week a learned model gets the same 30,000 rows, the same label, and a harder split. Two
questions, in order: **does it beat the rule**, and **does it beat it for a reason I can name**.
The second question is the lane's, and the answer to it is not the answer I expected.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Logistic Regression and a Random Forest, both ranked by predicted probability and scored at
precision@K — run in two feature arms.**

The baseline is a *ranked queue*, so the model has to produce a ranking, not a label: I take each
classifier's probability and score the ordering at the same K the baseline was frozen at.
Logistic Regression first because its coefficients are readable; Random Forest second because the
signals here are tiers and thresholds rather than smooth slopes, and if the extra capacity buys
nothing that is worth knowing. No boosting: the comparison below does not come close to earning it.

The part that fits *this* lane is the **two arms**, not the two algorithms:

- **Arm A** — observable page signals only (traffic, position, content, keyword demand).
- **Arm B** — arm A plus the generation metadata: `model_used`, `provider_used`, and a flag for
  whether a model was recorded at all.

ML-03 asked whether `model_used` adds signal once its confounds are controlled, using two features
and a random row split; it moved AUC by +0.0042. ML-07 asked it a second way and found the
most-flagged group was the one with *no model recorded*. Arm A vs arm B is the same question on a
third instrument: 42 features, client-grouped validation, a ranking metric. If the answer holds
across three instruments that fail differently, it is the dataset talking, not the method.

The classifier is still a measuring instrument here, not a product. Higher accuracy is not the goal.

The baseline is **re-encoded and recomputed in this run** below — not quoted from last week — so
every comparison in this notebook comes from one execution.

In [1]:
import os, sys, json
import numpy as np, pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-ml-capstone"
if IN_COLAB and not os.path.isdir("data/raw"):
    if not os.path.isdir(REPO_DIR):
        import subprocess
        subprocess.run(["git","clone","--depth","1",
                        f"https://github.com/ahmadhalawanii/{REPO_DIR}", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")
assert os.path.isdir("data/raw"), f"repo root not found from {os.getcwd()}"
os.makedirs("work/outputs", exist_ok=True)
pd.set_option("display.width", 190)

import sklearn
SEED = 42
print("working dir:", os.getcwd())
print(f"versions -- python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__} | sklearn {sklearn.__version__}")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
y      = df["is_declining_label"].values
groups = df["client_id"].values
imps   = df["impressions_90d"].values.astype(float)
BASE_RATE = y.mean()

# --- ML-07's rule, re-encoded here so the baseline is computed in THIS run, not quoted ---
FLOOR_IMPRESSIONS, POSITION_MAX = 100, 20
def tier_medians(frame):
    v = frame[(frame.avg_position > 0) & (frame.avg_position <= POSITION_MAX)
              & (frame.impressions_90d >= FLOOR_IMPRESSIONS)]
    return v.groupby("position_tier")["ctr"].median()

def baseline_score(frame, med):
    tm   = frame["position_tier"].map(med)
    elig = ((frame.avg_position > 0) & (frame.avg_position <= POSITION_MAX)
            & (frame.impressions_90d >= FLOOR_IMPRESSIONS) & tm.notna())
    gap  = (tm - frame["ctr"]).clip(lower=0).round(3)
    return np.where(elig & (gap > 0), gap, 0.0)

bl_global = baseline_score(df, tier_medians(df))

KS = [10, 20, 50, 100, 500]
def rank_desc(score, tie=None):
    tie = imps if tie is None else tie
    return np.lexsort((-tie, -score))
def prec_at_k(score, labels=None, tie=None, k=50):
    labels = y if labels is None else labels
    return float(labels[np.lexsort(((-imps if tie is None else -tie), -score))[:k]].mean())

print(f"\nrows {len(df):,} | clients {df.client_id.nunique()} | base rate {BASE_RATE:.3f}")
print("ML-07 baseline, recomputed in this run:")
for k in KS:
    print(f"  precision@{k:<4d}{prec_at_k(bl_global, k=k):.3f}")
assert abs(prec_at_k(bl_global, k=50) - 0.860) < 1e-9, "baseline drifted from the frozen ML-07 number"
print("\nmatches the frozen ML-07 receipt (precision@50 = 0.860). This is the number to beat.")

working dir: /content/flyrank-ml-capstone
versions -- python 3.13.15 | pandas 2.2.3 | numpy 2.1.3 | sklearn 1.6.1

rows 30,000 | clients 32 | base rate 0.542
ML-07 baseline, recomputed in this run:
  precision@10  0.800
  precision@20  0.800
  precision@50  0.860
  precision@100 0.880
  precision@500 0.824

matches the frozen ML-07 receipt (precision@50 = 0.860). This is the number to beat.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**`GroupKFold(5)` on `client_id`. No time split, and the reason is a limitation I have to state.**

Grouped by client because pages inside one client share a template, a CMS, a tracking setup, and
one SEO team's habits. On a random row split a model can learn *which client this is* and get
credit for it — and client base rates here run from 0.379 to 0.645, so knowing the client is worth
real AUC without knowing anything about content. Client sizes run from 3 rows to 7,008, and
`client_id` is a pseudonym: a grouping key, never a feature.

**No time split, because this snapshot has no time axis.** Every row is one trailing-90-day
observation; the label compares two windows *inside* that same snapshot. So this design answers
"does it transfer to a client I have not seen", not "does it transfer to next month". Temporal
transfer needs the warehouse panel, and that is ML-09's problem, not a claim I can make here.

**Two framings, both reported, because neither is clean on its own:**

- **pooled** — all 30,000 out-of-fold predictions ranked in one list. Matches the baseline's slice
  exactly, but mixes five models' probability scales into a single ordering; two pages ranked
  against each other may have been scored by different models.
- **per-fold** — each fold ranks only its own held-out rows, precision@50 averaged across folds.
  No cross-model scale problem, but K/N changes.

The rule is scored on the same folds with **fold-internal tier medians**, so both sides face the
same split. Its per-fold precision@50 comes out identical either way, which is itself a small
result: the tier medians are a stable global statistic, not a fitted thing that needs protecting.

In [2]:
from sklearn.model_selection import GroupKFold
gkf = GroupKFold(n_splits=5)
folds = list(gkf.split(df, y, groups))

rows = []
for f, (tr, te) in enumerate(folds):
    rows.append({"fold": f, "n_train": len(tr), "n_test": len(te),
                 "clients_held_out": int(pd.Series(groups[te]).nunique()),
                 "test_base_rate": round(float(y[te].mean()), 3),
                 "largest_test_client_share": round(float(pd.Series(groups[te]).value_counts(normalize=True).iloc[0]), 3)})
fold_table = pd.DataFrame(rows)
print("GroupKFold(5) grouped by client_id -- no client appears in train and test\n")
print(fold_table.to_string(index=False))
print(f"\nclient sizes: min {df.groupby('client_id').size().min()} rows, max {df.groupby('client_id').size().max():,} rows")
print(f"test-fold base rate spans {fold_table.test_base_rate.min():.3f} to {fold_table.test_base_rate.max():.3f} "
      f"(pooled {BASE_RATE:.3f}) -- clients are not exchangeable")

GroupKFold(5) grouped by client_id -- no client appears in train and test

 fold  n_train  n_test  clients_held_out  test_base_rate  largest_test_client_share
    0    22992    7008                 1           0.490                      1.000
    1    24269    5731                 7           0.645                      0.642
    2    24247    5753                 8           0.379                      0.399
    3    24245    5755                 8           0.622                      0.394
    4    24247    5753                 8           0.585                      0.312

client sizes: min 3 rows, max 7,008 rows
test-fold base rate spans 0.379 to 0.645 (pooled 0.542) -- clients are not exchangeable


**Fold 0 is one client with 7,008 rows.** `GroupKFold` balances folds by row count, so the largest
client fills a fold alone — a held-out set with no client diversity at all. I keep it rather than
reshuffling: it is the honest hard case, and section 4 reads its errors separately.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Same 30,000 rows, same label, same precision@K, same folds. Feature notes, all of them from
`docs/data-dictionary.md` and the `flyrank-data` gotcha list:

- `avg_position = 0` means **no position data**, so it becomes `NaN` with a `has_position` flag —
  never rank 0.
- `word_count` is missing for 28.3% of `keyword article` rows and 0% of everything else, so a blind
  `fillna(0)` would smuggle `content_type` in through the back door. Flags instead: `has_word_count`,
  `has_keyword_data`.
- The banned list from ML-07 is asserted against the feature frame, not eyeballed.
- Every arm gets `ctr_gap_pp` — **the baseline's own score** — as a feature, computed fold-internally.
  So the model is not competing with the rule; it starts from the rule and any gain is on top of it.

In [3]:
BANNED = ["trend_direction","trend_pct",
          "impressions_last_30d","clicks_last_30d","sessions_last_30d",
          "impressions_prev_30d","clicks_prev_30d","sessions_prev_30d"]

NUM = ["content_age_days","days_since_last_update","word_count","char_count",
       "search_volume","competition","cpc","impressions_90d","clicks_90d","pageviews_90d",
       "sessions_90d","users_90d","engaged_sessions_90d","ai_sessions_90d","scroll_events_90d",
       "days_with_impressions","days_with_sessions","ctr","engagement_rate","scroll_rate","ai_traffic_pct"]
CAT = ["content_type","main_intent","competition_level","position_tier"]
GEN = ["model_used","provider_used"]

b = df.copy()
b["avg_position_r"]   = b["avg_position"].replace(0, np.nan)   # 0 means "no data", never rank 0
b["has_position"]     = (df.avg_position > 0).astype(int)
b["has_word_count"]   = df.word_count.notna().astype(int)      # missingness follows content_type --
b["has_keyword_data"] = df.search_volume.notna().astype(int)   # flag it instead of fillna(0)

cat_block = pd.get_dummies(b[CAT].astype("object").fillna("(missing)"), prefix=CAT, dtype=float)
gen_block = pd.get_dummies(b[GEN].astype("object").fillna("(not recorded)"), prefix=GEN, dtype=float)
gen_block["has_model_recorded"] = df.model_used.notna().astype(int)
GEN_COLS = list(gen_block.columns)

X_num = b[NUM + ["avg_position_r","has_position","has_word_count","has_keyword_data"]].astype(float)
X_A = pd.concat([X_num, cat_block], axis=1)                 # arm A: observable page signals
X_B = pd.concat([X_A, gen_block], axis=1)                   # arm B: + generation metadata
X_C = X_A.drop(columns=["days_with_impressions","days_with_sessions"])  # arm C: see section 4

leaked = sorted(set(X_B.columns) & set(BANNED))
assert not leaked, f"LEAK: banned columns in feature frame: {leaked}"
assert not (set(["content_id","client_id"]) & set(X_B.columns)), "LEAK: an ID is being used as a feature"
print("arm A (page signals)          :", X_A.shape[1], "features")
print("arm B (+ generation metadata) :", X_B.shape[1], "features -- adds", GEN_COLS)
print("arm C (A minus 2 columns)     :", X_C.shape[1], "features -- built in section 4, listed here for one training pass")
print("\nPASS -- no banned column, no ID, in any arm.")

arm A (page signals)          : 42 features
arm B (+ generation metadata) : 52 features -- adds ['model_used_(not recorded)', 'model_used_gemini-2.5-flash', 'model_used_gemini-3-flash-preview', 'model_used_gpt-4o-mini', 'model_used_gpt-5-mini', 'model_used_unknown', 'provider_used_(not recorded)', 'provider_used_google', 'provider_used_openai', 'has_model_recorded']
arm C (A minus 2 columns)     : 40 features -- built in section 4, listed here for one training pass

PASS -- no banned column, no ID, in any arm.


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

def make_rf(): return RandomForestClassifier(n_estimators=300, min_samples_leaf=20,
                                             n_jobs=-1, random_state=SEED)
def make_lr(): return Pipeline([("impute", SimpleImputer(strategy="median")),
                                ("scale",  StandardScaler()),
                                ("clf",    LogisticRegression(max_iter=2000, random_state=SEED))])

ARMS = {"RF_A": (X_A, make_rf), "RF_B": (X_B, make_rf),
        "LR_A": (X_A, make_lr), "LR_B": (X_B, make_lr), "RF_C": (X_C, make_rf)}
oof     = {n: np.zeros(len(df)) for n in ARMS}
oof_bl  = np.zeros(len(df))
perm_rows, gen_block_drop = [], []

for f, (tr, te) in enumerate(folds):
    med = tier_medians(df.iloc[tr])                 # fitted on TRAIN clients only
    s_tr, s_te = baseline_score(df.iloc[tr], med), baseline_score(df.iloc[te], med)
    oof_bl[te] = s_te                               # the rule, scored on the same held-out rows
    for name, (X, mk) in ARMS.items():
        Xf = X.copy(); Xf["ctr_gap_pp"] = np.nan    # the baseline's own score, fold-internal
        c = Xf.columns.get_loc("ctr_gap_pp")
        Xf.iloc[tr, c], Xf.iloc[te, c] = s_tr, s_te
        m = mk().fit(Xf.iloc[tr], y[tr])
        oof[name][te] = m.predict_proba(Xf.iloc[te])[:, 1]
        if name == "RF_B":                          # permutation importance, held-out rows only
            Xte, yte = Xf.iloc[te], y[te]
            auc0 = roc_auc_score(yte, m.predict_proba(Xte)[:, 1])
            for j, col in enumerate(Xte.columns):
                r = np.random.RandomState(SEED + j)
                d = [auc0 - roc_auc_score(yte, m.predict_proba(Xte.assign(**{col: r.permutation(Xte[col].values)}))[:, 1])
                     for _ in range(3)]
                perm_rows.append({"fold": f, "feature": col, "auc_drop": float(np.mean(d))})
            r = np.random.RandomState(SEED)         # permute the WHOLE generation block together
            d = []
            for _ in range(3):
                Xp = Xte.copy(); Xp[GEN_COLS] = Xp[GEN_COLS].values[r.permutation(len(Xp))]
                d.append(auc0 - roc_auc_score(yte, m.predict_proba(Xp)[:, 1]))
            gen_block_drop.append(float(np.mean(d)))
    print(f"fold {f} trained  (held-out rows {len(te):,})")
print("\ndone -- every row now has a prediction from a model that never saw its client")

fold 0 trained  (held-out rows 7,008)
fold 1 trained  (held-out rows 5,731)
fold 2 trained  (held-out rows 5,753)
fold 3 trained  (held-out rows 5,755)
fold 4 trained  (held-out rows 5,753)

done -- every row now has a prediction from a model that never saw its client


In [5]:
def summarise(score):
    per_fold = [float(y[te][np.lexsort((-imps[te], -score[te]))[:50]].mean()) for _, te in folds]
    per_auc  = [float(roc_auc_score(y[te], score[te])) for _, te in folds]
    return {"auc_fold": float(np.mean(per_auc)), "p50_fold": float(np.mean(per_fold)),
            "p50_fold_min": min(per_fold), "p50_fold_max": max(per_fold),
            "p50_pooled": prec_at_k(score, k=50), "p100_pooled": prec_at_k(score, k=100),
            "p500_pooled": prec_at_k(score, k=500), "auc_pooled": float(roc_auc_score(y, score)),
            "distinct_scores": int(pd.Series(score).nunique())}

table = {"baseline (ML-07, global medians)": {**summarise(bl_global)},
         "baseline (fold-internal medians)": {**summarise(oof_bl)},
         "LR arm A": summarise(oof["LR_A"]), "LR arm B": summarise(oof["LR_B"]),
         "RF arm A": summarise(oof["RF_A"]), "RF arm B": summarise(oof["RF_B"])}
T = pd.DataFrame(table).T[["auc_fold","p50_fold","p50_fold_min","p50_fold_max",
                           "p50_pooled","p100_pooled","p500_pooled","distinct_scores"]].round(3)
T.columns = ["AUC(fold)","p@50(fold)","p@50 min","p@50 max","p@50(pool)","p@100(pool)","p@500(pool)","distinct scores"]
print(f"MODEL vs BASELINE -- 30,000 rows, label is_declining_label, base rate {BASE_RATE:.3f}")
print("client-grouped 5-fold; 'fold' = each fold ranks only its own held-out rows, averaged\n")
print(T.to_string())
print(f"\nbase rate {BASE_RATE:.3f} | ML-07 tie-break band for the rule: p@50 0.760-0.860")

MODEL vs BASELINE -- 30,000 rows, label is_declining_label, base rate 0.542
client-grouped 5-fold; 'fold' = each fold ranks only its own held-out rows, averaged

                                  AUC(fold)  p@50(fold)  p@50 min  p@50 max  p@50(pool)  p@100(pool)  p@500(pool)  distinct scores
baseline (ML-07, global medians)      0.584       0.836      0.76      0.92        0.86         0.88        0.824             30.0
baseline (fold-internal medians)      0.583       0.836      0.76      0.92        0.78         0.79        0.746             27.0
LR arm A                              0.650       0.788      0.70      0.92        0.68         0.76        0.776          29999.0
LR arm B                              0.614       0.724      0.60      0.80        0.62         0.66        0.740          29999.0
RF arm A                              0.674       0.816      0.60      0.92        0.60         0.71        0.782          29723.0
RF arm B                              0.672       0.

### The table

**The model does not beat the frozen number.**

- Rule: **0.836** per-fold precision@50 (**0.860** pooled, its ML-07 headline).
- Best model, RF arm A: **0.816** per-fold (**0.60** pooled).
- Base rate: 0.542. ML-07's tie-break band for the rule: 0.760–0.860.

RF arm A clears the base rate by a wide margin and lands inside the rule's band, but it does not
clear the top of it, which is what ML-07 pre-registered as the bar. On the frozen metric, the
answer is no.

**On whole-list ordering the model wins clearly, and the two results are not in conflict.**
AUC 0.674 vs 0.583 per fold — the model orders all 30,000 pages far better than the rule does. The
rule only has **27–30 distinct scores** to order 30,000 pages with (`ctr` ships rounded to two
decimals); the model has 29,723. The rule wins the top 50 and loses the list. The head and the list are
different questions, and precision@50 only ever answered the first one.

Two more things the table says out loud:

- **Random Forest > Logistic Regression** on both metrics (0.674 vs 0.650 AUC), but not by much.
  The signal here is thresholds and tiers, which is what trees are for — and a 0.024 AUC gap is not
  the kind of gap that would justify anything more complex.
- **Both arm Bs are worse than their arm As.** That is the next cell.

In [6]:
print("Does generation metadata add anything? (arm B minus arm A, same rows, same folds)\n")
for m in ["RF","LR"]:
    a, bb = summarise(oof[m+"_A"]), summarise(oof[m+"_B"])
    print(f"  {m}:  AUC {a['auc_fold']:.3f} -> {bb['auc_fold']:.3f}  ({bb['auc_fold']-a['auc_fold']:+.4f})"
          f"   p@50 {a['p50_fold']:.3f} -> {bb['p50_fold']:.3f}  ({bb['p50_fold']-a['p50_fold']:+.3f})")
print("\nPermuting the whole generation block inside arm B (AUC drop per fold; negative = model got BETTER):")
print("  ", [round(x, 4) for x in gen_block_drop])
print(f"   mean {np.mean(gen_block_drop):+.4f} | negative in {sum(1 for x in gen_block_drop if x < 0)} of {len(gen_block_drop)} folds")

Does generation metadata add anything? (arm B minus arm A, same rows, same folds)

  RF:  AUC 0.674 -> 0.672  (-0.0019)   p@50 0.816 -> 0.792  (-0.024)
  LR:  AUC 0.650 -> 0.614  (-0.0366)   p@50 0.788 -> 0.724  (-0.064)

Permuting the whole generation block inside arm B (AUC drop per fold; negative = model got BETTER):
   [-0.0011, 0.0036, -0.0079, -0.0244, -0.0116]
   mean -0.0083 | negative in 4 of 5 folds


**Verdict on the lane question: generation metadata does not add signal — it subtracts.**

Adding `model_used` + `provider_used` *lowers* held-out AUC in both algorithms (RF −0.0019,
LR −0.0366) and lowers per-fold precision@50 in both. Permuting the whole generation block inside
the fitted arm-B forest **improves** held-out AUC in 4 of 5 folds (mean −0.0083): shuffling which
model wrote a page makes the forest better at ranking it.

That is not noise-free — one fold goes the other way, and the RF gap is small enough that I would
not lean on it alone. But it is the third instrument to land in the same place: ML-03's +0.0042 AUC
(no lift), ML-07's flag-rate table (the most-flagged group was the one with no model recorded), and
now a fitted model that would rather not know. **Whatever `model_used` looks like it explains, it is
explaining it through content type, age, and metadata availability — not through the model.**

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### First: why the rule wins the top 50

ML-07 predicted this exact objection and pre-registered the answer. Here is what sits at the head of
each queue.

In [7]:
print("WHAT IS AT THE HEAD OF EACH QUEUE (top 50 rows by score)\n")
for name, s in [("baseline (global)", bl_global), ("baseline (fold)", oof_bl),
                ("RF arm A", oof["RF_A"]), ("LR arm A", oof["LR_A"])]:
    idx = rank_desc(s)[:50]; t = df.iloc[idx]
    print(f"{name:<20} p@50 {y[idx].mean():.3f} | zero clicks {(t.clicks_90d==0).mean():.0%} | ctr==0 {(t.ctr==0).mean():.0%} "
          f"| median impressions {int(t.impressions_90d.median()):>6,} | tiers {dict(t.position_tier.value_counts())}")

WHAT IS AT THE HEAD OF EACH QUEUE (top 50 rows by score)

baseline (global)    p@50 0.860 | zero clicks 98% | ctr==0 100% | median impressions  4,172 | tiers {'page_1': np.int64(50)}
baseline (fold)      p@50 0.780 | zero clicks 100% | ctr==0 100% | median impressions  2,488 | tiers {'page_1': np.int64(50)}
RF arm A             p@50 0.600 | zero clicks 94% | ctr==0 94% | median impressions    295 | tiers {'page_1': np.int64(35), 'striking': np.int64(9), 'page_3_5': np.int64(4), 'top_3': np.int64(2)}
LR arm A             p@50 0.680 | zero clicks 68% | ctr==0 68% | median impressions  1,190 | tiers {'page_1': np.int64(41), 'striking': np.int64(6), 'page_3_5': np.int64(2), 'top_3': np.int64(1)}


**The model did not escape the zero-click block. It rediscovered it.**

The rule's top 50 is 100% zero-CTR `page_1` pages — the block ML-07 diagnosed as broken click
tracking rather than weak titles. The forest, given 42 features and no instruction about which rows
are actionable, filled **94%** of its own top 50 with the same kind of row, at a *lower* impression
level (median 295 vs 4,172) where the CTR estimate is noisier still.

So the rule's precision@50 advantage is not skill the model lacks. Both queues are ranking the same
artifact; the rule just concentrates on it harder — and the block pays off because of exactly where
it sits. Across all 30,000 rows, zero-click pages decline at **0.496**, *below* the 0.542 base rate.
Zero-click pages **inside the rule's eligibility gate** (top-20 rank, at least 100 impressions)
decline at **0.756** — ML-07's figure, same slice. The head of the queue is not "pages with no
clicks"; it is a specific intersection the rule's own gate constructed, and precision@50 is
measuring that construction. **A metric can be won by leaning further into the thing your error
analysis told you not to trust.**

Logistic Regression leans on it least — 68% of its top 50, the only queue head here that is not
almost entirely one block.

### What the model leans on

Permutation importance on held-out rows, not the fitted-on-train importances — a feature that only
looks important in-sample is not important.

In [8]:
P = pd.DataFrame(perm_rows).groupby("feature").auc_drop.mean().sort_values(ascending=False)
print("PERMUTATION IMPORTANCE -- RF arm B, mean AUC drop across the 5 held-out folds\n")
print("top 8:");  print(P.head(8).round(4).to_string())
print("\nbottom 5:"); print(P.tail(5).round(4).to_string())
print("\ngeneration-metadata columns, ranked among all", len(P), "features:")
print(P[P.index.isin(GEN_COLS)].round(4).to_string())

PERMUTATION IMPORTANCE -- RF arm B, mean AUC drop across the 5 held-out folds

top 8:
feature
days_with_impressions    0.0427
impressions_90d          0.0140
avg_position_r           0.0124
ctr_gap_pp               0.0118
content_age_days         0.0091
clicks_90d               0.0045
scroll_rate              0.0044
ctr                      0.0038

bottom 5:
feature
char_count                    -0.0016
provider_used_google          -0.0018
model_used_gemini-2.5-flash   -0.0018
word_count                    -0.0035
model_used_gpt-4o-mini        -0.0046

generation-metadata columns, ranked among all 53 features:
feature
model_used_gpt-5-mini                0.0001
model_used_unknown                   0.0000
has_model_recorded                   0.0000
model_used_(not recorded)            0.0000
provider_used_openai                -0.0003
model_used_gemini-3-flash-preview   -0.0004
provider_used_(not recorded)        -0.0008
provider_used_google                -0.0018
model_used_gemini-2.5

`days_with_impressions` is the top feature by a factor of three over anything else, and that is
the number worth stopping on. ML-07 killed the staleness signal by showing its gradient was a
**labelability artifact**: rows with `impressions_prev_30d = 0` have no denominator, are scored
`new`/`flat`, and *can never carry the `down` label at all*. So I ran the model's favourite feature
through the same check.

In [9]:
df["label_eligible"] = df.impressions_prev_30d > 0
bins = pd.cut(df.days_with_impressions, [0, 7, 30, 60, 90], labels=["1-7","8-30","31-60","61-90"])
t = (df.assign(bucket=bins).groupby("bucket", observed=True)
       .agg(n=("is_declining_label","size"), pct_label_eligible=("label_eligible","mean"),
            decline_rate=("is_declining_label","mean")).round(3))
t["decline_rate_eligible_only"] = (df[df.label_eligible].assign(bucket=bins)
                                   .groupby("bucket", observed=True).is_declining_label.mean().round(3))
print("The model's strongest feature, checked the way ML-07 checked staleness:\n")
print(t.to_string())
print(f"\nspread across buckets -- all rows: {t.decline_rate.max()-t.decline_rate.min():.3f}"
      f"   label-eligible rows only: {t.decline_rate_eligible_only.max()-t.decline_rate_eligible_only.min():.3f}")
print(f"corr(days_with_impressions, label is even possible) = {df.days_with_impressions.corr(df.label_eligible.astype(int)):.3f}")

elig = df.label_eligible.values
print(f"\nSame models, scored on label-eligible rows only (n={elig.sum():,}, base rate {y[elig].mean():.3f}):")
for name, s in [("baseline (global)", bl_global), ("RF arm A", oof["RF_A"]), ("RF arm C", oof["RF_C"])]:
    print(f"  {name:<20} AUC {roc_auc_score(y[elig], s[elig]):.3f}   p@50 {prec_at_k(s[elig], y[elig], imps[elig], 50):.3f}")

print("\nARM C -- drop days_with_impressions and days_with_sessions and refit:")
a, c = summarise(oof["RF_A"]), summarise(oof["RF_C"])
print(f"  AUC(fold) {a['auc_fold']:.3f} -> {c['auc_fold']:.3f} ({c['auc_fold']-a['auc_fold']:+.4f})   "
      f"p@50(fold) {a['p50_fold']:.3f} -> {c['p50_fold']:.3f} ({c['p50_fold']-a['p50_fold']:+.3f})")
print(f"  permutation said days_with_impressions alone was worth {P['days_with_impressions']:.4f} AUC;")
print(f"  removing it and its sibling costs {a['auc_fold']-c['auc_fold']:.4f}. Correlated columns absorb it.")

The model's strongest feature, checked the way ML-07 checked staleness:

            n  pct_label_eligible  decline_rate  decline_rate_eligible_only
bucket                                                                     
1-7      3985               0.311         0.210                       0.674
8-30     3450               0.817         0.528                       0.647
31-60    3284               0.997         0.665                       0.668
61-90   19281               1.000         0.592                       0.592

spread across buckets -- all rows: 0.455   label-eligible rows only: 0.082
corr(days_with_impressions, label is even possible) = 0.616

Same models, scored on label-eligible rows only (n=26,612, base rate 0.611):
  baseline (global)    AUC 0.555   p@50 0.860
  RF arm A             AUC 0.616   p@50 0.600
  RF arm C             AUC 0.614   p@50 0.700

ARM C -- drop days_with_impressions and days_with_sessions and refit:
  AUC(fold) 0.674 -> 0.664 (-0.0096)   p@50(fold

**Verdict: the model's strongest feature is a label-eligibility proxy.**

Across `days_with_impressions` buckets the decline rate spreads by **0.455** — enormous, the kind of
signal that looks like a finding. Among rows that can actually carry the label, it spreads by
**0.082** and stops being monotone. The `1-7` bucket reads 0.210 declining on all rows and 0.674 on
eligible rows: 69% of those pages were never eligible to be labelled `down`, and their presence in
the denominator is the entire gradient. Correlation between `days_with_impressions` and "the label
is even possible for this row" is **0.616**.

This is the same failure ML-07 found in `freshness_tier` (0.140 → 0.033), in a different column, found
by a model instead of by hand. **The forest's best feature is not measuring decline. It is measuring
which rows the label definition allowed to be positive.**

It is not leakage in the ML-05 sense — `days_with_impressions` is knowable before the outcome and
uses no banned column. It is worse in one specific way: leakage announces itself with a suspiciously
high score, and this does not. It sits at AUC 0.674 looking like an ordinary, modest, believable model.

**And you cannot fix it by dropping the column** (arm C, below). Removing `days_with_impressions` and
`days_with_sessions` costs only 0.0096 AUC, against the 0.0427 permutation said the first column alone
was worth — because `impressions_90d`, `clicks_90d`, `sessions_90d` and the rest carry the same
eligibility structure. Permutation importance measures a column's *marginal* value inside one fitted
model; with correlated substitutes present, that is not what removing it costs. The artifact lives in
the **label definition**, so it cannot be quarantined by deleting features. The honest fixes are to
score on eligible rows only, or to redefine the label — and on eligible rows the model's edge over
the rule survives (AUC 0.616 vs 0.555), which is the one reassuring number in this section.

### Calibration, error groups, and three confident mistakes

In [10]:
print("CALIBRATION -- RF arm A, out-of-fold predictions in deciles\n")
d = pd.DataFrame({"p": oof["RF_A"], "y": y}); d["decile"] = pd.qcut(d.p, 10, labels=False)
print(d.groupby("decile").agg(n=("y","size"), mean_predicted=("p","mean"), actual_rate=("y","mean")).round(3).to_string())

pred = (oof["RF_A"] >= 0.5).astype(int); df["_correct"] = (pred == y)
print("\nWHERE IT IS WRONG (accuracy read next to the group's own decline rate -- accuracy alone lies where the mix is skewed)\n")
for col in ["content_type","position_tier","impression_tier"]:
    print(df.groupby(col).agg(n=("_correct","size"), accuracy=("_correct","mean"),
                              actual_decline=("is_declining_label","mean")).round(3).to_string(), "\n")
cl = df.groupby("client_id").agg(n=("_correct","size"), accuracy=("_correct","mean"),
                                 actual_decline=("is_declining_label","mean")).round(3)
print("hardest clients (>=200 rows):"); print(cl[cl.n >= 200].sort_values("accuracy").head(3).to_string())

print("\nTHREE CONFIDENT MISTAKES -- ranked near the top by RF arm A, not actually declining\n")
order = rank_desc(oof["RF_A"]); shown = 0
for pos, i in enumerate(order[:300]):
    if y[i] == 1 or shown == 3: continue
    r = df.iloc[i]; shown += 1
    print(f"queue rank {pos+1:>3}  p={oof['RF_A'][i]:.3f}  actual trend: {r.trend_direction}")
    print(f"   ranks {r.avg_position} ({r.position_tier}), ctr {r.ctr}%, {int(r.impressions_90d):,} impressions, "
          f"{int(r.clicks_90d)} clicks, {int(r.sessions_90d)} sessions, {r.content_type}")
    print(f"   why it is hard: impressions went {int(r.impressions_prev_30d):,} -> {int(r.impressions_last_30d):,} "
          f"(that is the label, unseen by the model), and with 0 recorded clicks the model has no CTR evidence either way")

CALIBRATION -- RF arm A, out-of-fold predictions in deciles

           n  mean_predicted  actual_rate
decile                                   
0       3000           0.186        0.145
1       3000           0.364        0.370
2       3000           0.449        0.480
3       3000           0.511        0.545
4       3000           0.560        0.572
5       3000           0.601        0.583
6       3000           0.639        0.616
7       3000           0.678        0.644
8       3000           0.725        0.694
9       3000           0.803        0.772

WHERE IT IS WRONG (accuracy read next to the group's own decline rate -- accuracy alone lies where the mix is skewed)

                        n  accuracy  actual_decline
content_type                                       
comparison article    697     0.574           0.572
feedly article       2096     0.766           0.287
keyword article     27207     0.633           0.561 

                   n  accuracy  actual_decline
positi

**Reading those:**

- **Calibration is decent and slightly over-confident at the top** — the top decile predicts 0.803 and
  delivers 0.772. Nothing is pathological; the ordering is real even where the reason for it is not.
- **Accuracy by group is a trap and I am reporting it next to each group's own decline rate for that
  reason.** `top_3` shows 0.890 accuracy — its decline rate is 0.241, so predicting "not declining"
  everywhere would score 0.759 there. `feedly article` is the same story (0.766 accuracy, 0.287
  decline rate). The genuinely hard group is `striking` (0.599 accuracy at a 0.610 decline rate) —
  close to a coin flip on the tier where an SEO team would most want an answer, because pages on the
  edge of page 1 move for reasons this snapshot does not record.
- **The hardest clients are hardest at the client level, not the page level.** The worst client sits
  at 0.425 accuracy — below its own base rate. Client-grouped validation is what exposes this; a
  random split would have averaged it away.
- **All three confident mistakes are the same mistake.** Every one is a zero-click, zero-CTR `page_1`
  page that the model ranked near the top, and every one actually went *up* or stayed flat. With no
  recorded clicks there is no CTR evidence in either direction, so the model is scoring the shape of
  the row rather than anything about its trajectory. These are not near-misses; they are the queue's
  head being un-decidable from the features available.

### The pre-registered test: does the model win where the rule was actionable?

ML-07 declared this before seeing any model result: exclude pages with zero recorded clicks in 90
days — the rows its top-20 review found un-actionable — and see whether the ordering holds. Running
it now, on a hypothesis registered last week rather than one found by looking, is the only version
of this test that means anything.

In [11]:
mask = (df.clicks_90d > 0).values
print("ML-07 pre-registered this test: exclude pages with zero recorded clicks in 90 days --")
print("the rows its own top-20 review found un-actionable (a title rewrite cannot fix broken click tracking).\n")
print(f"slice: {mask.sum():,} of {len(df):,} pages, base rate {y[mask].mean():.3f}\n")
res_rows = []
for name, s in [("baseline (global)", bl_global), ("baseline (fold)", oof_bl),
                ("LR arm A", oof["LR_A"]), ("RF arm A", oof["RF_A"]), ("RF arm C", oof["RF_C"])]:
    res_rows.append({"ranking": name,
                     "AUC": round(float(roc_auc_score(y[mask], s[mask])), 3),
                     **{f"p@{k}": round(prec_at_k(s[mask], y[mask], imps[mask], k), 3) for k in [10,20,50,100,500]}})
print(pd.DataFrame(res_rows).set_index("ranking").to_string())
print(f"\nbase rate on this slice {y[mask].mean():.3f}")

ML-07 pre-registered this test: exclude pages with zero recorded clicks in 90 days --
the rows its own top-20 review found un-actionable (a title rewrite cannot fix broken click tracking).

slice: 16,796 of 30,000 pages, base rate 0.578

                     AUC  p@10  p@20  p@50  p@100  p@500
ranking                                                 
baseline (global)  0.535   0.7  0.70  0.76   0.71  0.672
baseline (fold)    0.529   0.6  0.65  0.64   0.71  0.660
LR arm A           0.582   0.4  0.60  0.66   0.72  0.718
RF arm A           0.609   0.6  0.70  0.78   0.78  0.766
RF arm C           0.603   0.8  0.65  0.70   0.73  0.768

base rate on this slice 0.578


**On the actionable slice the model wins, and by the margin the headline metric was hiding.**

| | AUC | p@50 | p@100 | p@500 |
|---|---|---|---|---|
| rule (global) | 0.535 | 0.760 | 0.710 | 0.672 |
| **RF arm A** | **0.609** | **0.780** | **0.780** | **0.766** |

The rule's AUC falls to 0.535 — barely above chance — once the zero-click block is removed. Its
precision@50 falls from 0.860 to 0.760. The model beats it at every K on this slice — 0.780 at K=50,
0.780 at K=100, 0.766 at K=500, where the rule drops to 0.710 and 0.672.

**So the honest headline is a split decision, and both halves matter:**

> On the frozen slice and the frozen metric, the rule wins (0.836 vs 0.816 per-fold) — and its win is
> concentrated in rows its own error analysis said were not actionable. On the rows a human can act
> on, the model wins on every measure. Measured on this slice, the model is
> decision-support the rule is not; it is not a better queue by the number the rule was frozen on.

Nothing here got retuned to produce that. Arm C — the same model minus the two eligibility-proxy
columns — lands in the same place (AUC 0.603, p@50 0.700), so the result does not depend on the
artifact either.

### The receipt

In [12]:
metrics = {
 "notebook": "w05_model.ipynb", "assignment": "ML-08", "lane": "Freestyle: Confound or Cause?",
 "data_slice": "data/raw/content_refresh_anonymized.csv (30,000 rows, 32 clients)",
 "label": "is_declining_label = (trend_direction == 'down')",
 "base_rate": round(float(BASE_RATE), 4), "seed": SEED,
 "versions": {"pandas": pd.__version__, "numpy": np.__version__, "sklearn": sklearn.__version__},
 "split": {"design": "GroupKFold(5) on client_id", "folds": fold_table.to_dict("records")},
 "comparison": {k: {kk: round(vv, 4) for kk, vv in v.items()} for k, v in table.items()},
 "generation_metadata": {"block_permutation_auc_drop_per_fold": [round(x,4) for x in gen_block_drop],
                         "mean": round(float(np.mean(gen_block_drop)), 4)},
 "top_permutation_features": P.head(5).round(4).to_dict(),
 "eligibility_artifact": {"feature": "days_with_impressions",
                          "decline_spread_all_rows": round(float(t.decline_rate.max()-t.decline_rate.min()), 3),
                          "decline_spread_eligible_only": round(float(t.decline_rate_eligible_only.max()-t.decline_rate_eligible_only.min()), 3)},
 "actionable_slice": {"definition": "clicks_90d > 0", "n": int(mask.sum()),
                      "base_rate": round(float(y[mask].mean()), 4),
                      "results": res_rows},
}
with open("work/outputs/model_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("wrote work/outputs/model_metrics.json -- the receipt every number above traces back to")

wrote work/outputs/model_metrics.json -- the receipt every number above traces back to


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] Compared against the Week-4 baseline **on the same 30,000 rows, the same label, the same
      precision@K, and the same folds** — with the baseline re-encoded and recomputed in this run,
      not quoted
- [x] Valid validation design: `GroupKFold(5)` on `client_id`, no client in train and test; the
      absence of a time split is stated as a limitation, not hidden
- [x] Method choice explained and kept as simple as the comparison earns — no boosting added for a
      gap that would not have justified it
- [x] Features interpreted by permutation importance on held-out rows, and the top feature
      **failed** the check I ran it through
- [x] Errors read before the score was believed: queue-head composition, calibration deciles, error
      groups next to their own base rates, worst clients, three concrete confident mistakes
- [x] Seed fixed (42) and library versions printed; `GroupKFold` is deterministic, so a fresh clone
      reproduces the table
- [x] Metrics receipt committed to `work/outputs/model_metrics.json`
- [x] No client names, URLs, or private queries anywhere
- [x] Careful words throughout: observed, measured, directional, decision-support

### What I would tell FlyRank, in four lines

1. The Week-4 rule still owns the top of the queue; the model orders the *whole list* far better
   (AUC 0.674 vs 0.583) and wins outright on the pages a human can actually act on.
2. Neither ranking should be shipped as-is: both fill their head with pages whose clicks are not
   being recorded, and that is a tracking ticket, not a content ticket.
3. The model's strongest feature turned out to measure which rows the label *allowed* to be
   positive, not which pages declined. Every number in this notebook is reported next to that.
4. Generation metadata still adds nothing — third instrument, same answer.

### Handing forward to ML-09

Two things to attack next week, both named here rather than discovered later:

- **The label definition is the bottleneck, not the model.** `impressions_prev_30d = 0` rows cannot
  be positive, and that single construction rule produced the false staleness signal in ML-07 and
  the false top feature here. ML-09 should test whether restricting to label-eligible rows — or
  redefining decline so every row can carry it — changes the model-vs-rule verdict.
- **Fold 0 is one client.** Client-level accuracy ranges from 0.425 to well above average, so a
  single pooled number is hiding a spread. Per-client validation, not just per-fold.